# DNA-DetectLLM Demo: R(s) vs R_final

**IE 663 End-term -- Team Eternal (Tanmay Mandaliya, 22B1037)**

This notebook demonstrates the vulnerability of DNA-DetectLLM's R(s) score to humanization attacks, and how our proposed R_final extension catches them.

**Pipeline:**
1. Load Falcon-7B (observer) and Falcon-7B-Instruct (performer)
2. Score a **human** text, an **AI** text, and a **humanized** text with R(s) and R_final
3. Show that R(s) is fooled by humanization, while R_final catches it

**Requirements:** Kaggle GPU (P100 or T4), ~14 GB VRAM with 4-bit quantization.

In [ ]:
# Cell 2: Install dependencies and load models
import subprocess, sys, os

# Install exact versions matching our Kaggle experiments
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'torch==2.5.1+cu121', 'torchvision==0.20.1+cu121',
    '--index-url', 'https://download.pytorch.org/whl/cu121', '-q'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'transformers==4.46.3', 'accelerate==1.1.1', 'bitsandbytes',
    'scipy', 'scikit-learn', 'requests', '-q'])

import gc, warnings, ctypes
import numpy as np
import torch
import torch.nn.functional as F
warnings.filterwarnings('ignore')

print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, compute {p.major}.{p.minor}, {p.total_mem/1e9:.1f} GB')

# Load tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
DEVICE = 'cuda:0'
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained('tiiuae/falcon-7b')
tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization config (works on both P100 and T4)
cc = torch.cuda.get_device_properties(0)
if (cc.major, cc.minor) >= (7, 0):
    qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    lk = dict(quantization_config=qc, device_map='auto', low_cpu_mem_usage=True)
    MAX_LEN = 512
else:
    lk = dict(torch_dtype=torch.float16, device_map='auto', low_cpu_mem_usage=True)

# Load observer (Falcon-7B) and performer (Falcon-7B-Instruct)
print('Loading observer (Falcon-7B)...')
observer = AutoModelForCausalLM.from_pretrained('tiiuae/falcon-7b', **lk)
observer.eval()
gc.collect(); torch.cuda.empty_cache()

print('Loading performer (Falcon-7B-Instruct)...')
performer = AutoModelForCausalLM.from_pretrained('tiiuae/falcon-7b-instruct', **lk)
performer.eval()
gc.collect(); torch.cuda.empty_cache()
try: ctypes.CDLL('libc.so.6').malloc_trim(0)
except: pass

print(f'\nBoth models loaded. MAX_LEN={MAX_LEN}')

In [ ]:
# Cell 3: Define compute_all() -- returns R(s) + three auxiliary features
# This is the same function used in all our Kaggle scoring notebooks.

@torch.inference_mode()
def compute_all(text):
    """
    Score a single text with DNA-DetectLLM R(s) and auxiliary features.
    
    Returns dict with:
        rs          : R(s) = (PPL + PPL_hat) / (2 * X-PPL)     -- original score
        ce_var      : Variance of per-token cross-entropy        -- Feature 1
        agree_rate  : Observer-performer argmax agreement rate    -- Feature 2
        coherence   : mean_CE_confident - mean_CE_uncertain      -- Feature 3
    """
    torch.cuda.empty_cache()
    
    # Tokenize
    enc = tokenizer(text, return_tensors='pt', truncation=True,
                    max_length=MAX_LEN, return_token_type_ids=False)
    eg = {k: v.to(DEVICE) for k, v in enc.items()}
    
    # Forward pass through both models (single pass each)
    obs_logits = observer(**eg).logits.float()    # observer: Falcon-7B
    perf_logits = performer(**eg).logits.float()  # performer: Falcon-7B-Instruct
    
    # --- R(s) computation ---
    # Shifted logits for next-token prediction
    shifted = perf_logits[..., :-1, :]
    labels = eg['input_ids'][..., 1:]
    attn = eg['attention_mask'][..., 1:]
    
    # Per-token cross-entropy from performer
    ce = F.cross_entropy(shifted.transpose(1, 2), labels, reduction='none')
    S = ce[attn.bool()].cpu().numpy()  # shape: (L,)
    
    if len(S) < 5:
        del obs_logits, perf_logits, eg
        return None
    
    # PPL(s) = mean CE at actual tokens
    ppl_s = float(np.mean(S))
    
    # Ideal tokens = argmax at each position from performer
    ideal_tokens = shifted.argmax(dim=-1)  # (1, L)
    
    # PPL(s_hat|s) = mean CE at ideal tokens
    # (CE of ideal tokens is just -log P(argmax), which = min CE)
    perf_probs = F.softmax(shifted, dim=-1)
    ideal_ce = -torch.log(perf_probs.gather(2, ideal_tokens.unsqueeze(-1)).squeeze(-1) + 1e-10)
    ppl_hat = float(ideal_ce[attn.bool()].mean().cpu().item())
    
    # X-PPL = cross-perplexity between observer and performer
    obs_shifted = obs_logits[..., :-1, :]
    obs_log_probs = F.log_softmax(obs_shifted, dim=-1)
    perf_probs_full = F.softmax(shifted, dim=-1)
    xent = -(perf_probs_full * obs_log_probs).sum(dim=-1)
    x_ppl = float(xent[attn.bool()].mean().cpu().item())
    
    # R(s) = (PPL + PPL_hat) / (2 * X-PPL)
    rs = (ppl_s + ppl_hat) / (2 * x_ppl) if x_ppl > 1e-6 else 0.5
    
    # --- Feature 1: CE Variance ---
    ce_var = float(np.var(S))
    
    # --- Feature 2: Observer-Performer Agreement Rate ---
    obs_top = obs_shifted.argmax(dim=-1)
    perf_top = shifted.argmax(dim=-1)
    agree = (obs_top == perf_top).float()[attn.bool()].cpu().numpy()
    agree_rate = float(np.mean(agree))
    
    # --- Feature 3: Confidence-Coherence Gap ---
    # Performer entropy at each position
    pe = -(F.softmax(shifted, dim=-1) * F.log_softmax(shifted, dim=-1)).sum(-1)
    H = pe[attn.bool()].cpu().numpy()
    H_med = np.median(H)
    conf_mask = H < H_med  # confident = low entropy
    if conf_mask.sum() > 0 and (~conf_mask).sum() > 0:
        coherence = float(np.mean(S[conf_mask]) - np.mean(S[~conf_mask]))
    else:
        coherence = 0.0
    
    # Count mutations for display
    n_mutations = int((labels[attn.bool()] != ideal_tokens[attn.bool()]).sum().cpu().item())
    n_total = int(attn.bool().sum().cpu().item())
    
    del obs_logits, perf_logits, eg
    return {
        'rs': rs,
        'ce_var': ce_var,
        'agree_rate': agree_rate,
        'coherence': coherence,
        'n_mutations': n_mutations,
        'n_total': n_total,
    }

# Quick sanity check
test = compute_all("Hello world, this is a test sentence for the detector.")
print(f"Sanity check: R(s)={test['rs']:.4f}, CE-var={test['ce_var']:.2f}, "
      f"agree={test['agree_rate']:.3f}, coh={test['coherence']:.2f}, "
      f"mutations={test['n_mutations']}/{test['n_total']}")

In [ ]:
# Cell 4: Define R_final with hardcoded human baselines from our experiments
# Baselines computed from 800 XSum human texts scored by Falcon-7B detector.

# Human baseline statistics (from pooled benchmark experiments)
CEV_MU, CEV_SIGMA = 6.24, 1.31    # CE variance: mean, std
AGR_MU, AGR_SIGMA = 0.739, 0.066  # Agreement rate: mean, std
COH_MU, COH_SIGMA = -2.41, 0.52   # Coherence gap: mean, std

def R_final(scores, k=1.0):
    """
    Compute R_final = R(s) * P1(ce_var) * P2(agree) * P3(coherence)
    
    Each penalty fires only when the feature deviates beyond k standard
    deviations from the human baseline. This means:
    - Normal human text:  all penalties ~1.0, R_final ~ R(s)
    - Normal AI text:     all penalties ~1.0, R_final ~ R(s) (still low)
    - Humanized text:     penalties < 1.0,    R_final << R(s) (caught!)
    
    Args:
        scores: dict from compute_all()
        k: sensitivity parameter (std devs beyond human mean before penalty)
    Returns:
        dict with rs, r_final, and individual penalties
    """
    rs = scores['rs']
    
    # P1: Penalize HIGH CE variance (humanization creates bimodal CE pattern)
    # Humanized text: function words have low CE, replaced content words have very high CE
    cev_excess = max(0, scores['ce_var'] - (CEV_MU + k * CEV_SIGMA))
    p1 = max(0.1, 1 - 0.1 * cev_excess)
    
    # P2: Penalize LOW agreement rate (humanized tokens diverge from observer's predictions)
    # When many tokens are replaced, observer and performer disagree more
    agr_deficit = max(0, (AGR_MU - k * AGR_SIGMA) - scores['agree_rate'])
    p2 = max(0.1, 1 - 3.0 * agr_deficit)
    
    # P3: Penalize abnormally NEGATIVE coherence
    # Humanized text: confident positions have surprisingly wrong tokens
    coh_deficit = max(0, (COH_MU - k * COH_SIGMA) - scores['coherence'])
    p3 = max(0.1, 1 - 0.5 * coh_deficit)
    
    r_final = rs * p1 * p2 * p3
    
    return {
        'rs': rs,
        'r_final': r_final,
        'p1_cevar': p1,
        'p2_agree': p2,
        'p3_coherence': p3,
        'ce_var': scores['ce_var'],
        'agree_rate': scores['agree_rate'],
        'coherence': scores['coherence'],
        'mutations': f"{scores['n_mutations']}/{scores['n_total']}",
    }

# Thresholds
TAU_RS = 0.64      # R(s) threshold (from 4-bit Falcon calibration)
TAU_RFINAL = 0.50  # R_final threshold (lower because penalties reduce scale)

def classify(result):
    """Return verdict string for both R(s) and R_final."""
    rs_label = "Human" if result['rs'] > TAU_RS else "AI"
    rf_label = "Human" if result['r_final'] > TAU_RFINAL else "AI/Humanized"
    return rs_label, rf_label

print(f"Human baselines: CE-var={CEV_MU}+/-{CEV_SIGMA}, "
      f"agree={AGR_MU}+/-{AGR_SIGMA}, coh={COH_MU}+/-{COH_SIGMA}")
print(f"Thresholds: R(s) tau={TAU_RS}, R_final tau={TAU_RFINAL}")
print("R_final function ready.")

In [ ]:
# Cell 5: Score a HUMAN text (from XSum dataset)

HUMAN_TEXT = (
    "Two special concerts at the Royal Albert Hall will see pupils perform "
    "their own response to 10 pieces of classic music. The Ten Pieces project "
    "was announced last year as part of an initiative to inspire primary school "
    "children to learn more about classical music. Children from across the UK "
    "have been creating their own artistic responses to the pieces chosen by "
    "the BBC, ranging from dance routines to digital animations."
)

print("=" * 70)
print("HUMAN TEXT (XSum):")
print(f"  \"{HUMAN_TEXT[:120]}...\"")
print(f"  Length: {len(HUMAN_TEXT.split())} words")
print()

human_raw = compute_all(HUMAN_TEXT)
human_result = R_final(human_raw)
rs_verdict, rf_verdict = classify(human_result)

print(f"  R(s)      = {human_result['rs']:.4f}  -->  {rs_verdict} {'[correct]' if rs_verdict == 'Human' else '[WRONG]'}")
print(f"  R_final   = {human_result['r_final']:.4f}  -->  {rf_verdict} {'[correct]' if rf_verdict == 'Human' else '[WRONG]'}")
print(f"  Mutations = {human_result['mutations']}")
print(f"  Penalties: P1(CE-var)={human_result['p1_cevar']:.3f}, "
      f"P2(agree)={human_result['p2_agree']:.3f}, P3(coh)={human_result['p3_coherence']:.3f}")
print(f"  Features:  CE-var={human_result['ce_var']:.2f}, "
      f"agree={human_result['agree_rate']:.3f}, coh={human_result['coherence']:.2f}")
print("=" * 70)

In [ ]:
# Cell 6: Score an AI text (GPT-4 generated, same topic)

AI_TEXT = (
    "The concerts, set to be held on two separate days this month, are part of "
    "an ambitious project spearheaded by music educators from across the country. "
    "The project encourages students from a wide age range and broad spectrum of "
    "musical abilities to engage with, and perform their interpretations of, 10 "
    "notable pieces from the classical music canon. The 10 pieces of music, "
    "carefully curated by renowned orchestra conductors and music teachers, include "
    "works from celebrated composers such as Beethoven, Mozart, Chopin, and "
    "Tchaikovsky. The selection is designed to expose students to various musical "
    "styles and historical periods, fostering a deeper appreciation for classical music."
)

print("=" * 70)
print("AI TEXT (GPT-4):")
print(f"  \"{AI_TEXT[:120]}...\"")
print(f"  Length: {len(AI_TEXT.split())} words")
print()

ai_raw = compute_all(AI_TEXT)
ai_result = R_final(ai_raw)
rs_verdict, rf_verdict = classify(ai_result)

print(f"  R(s)      = {ai_result['rs']:.4f}  -->  {rs_verdict} {'[correct]' if rs_verdict == 'AI' else '[WRONG]'}")
print(f"  R_final   = {ai_result['r_final']:.4f}  -->  {rf_verdict} {'[correct]' if 'AI' in rf_verdict else '[WRONG]'}")
print(f"  Mutations = {ai_result['mutations']}")
print(f"  Penalties: P1(CE-var)={ai_result['p1_cevar']:.3f}, "
      f"P2(agree)={ai_result['p2_agree']:.3f}, P3(coh)={ai_result['p3_coherence']:.3f}")
print(f"  Features:  CE-var={ai_result['ce_var']:.2f}, "
      f"agree={ai_result['agree_rate']:.3f}, coh={ai_result['coherence']:.2f}")
print("=" * 70)

In [ ]:
# Cell 7: Show the P_LIT3 humanization prompt
# This is our strongest literature-based prompt (combined vocabulary + structure attack)

P_LIT3 = """Rewrite this text so it reads like a specific human wrote it on a tight deadline -- not like an AI responding to a prompt.

VOCABULARY RULES:
- Replace every formal/academic word with a vivid informal alternative. Don't just simplify -- use SPECIFIC, COLORFUL language. Example: 'significant increase' -> 'a sharp spike'; 'implemented a solution' -> 'patched the mess'; 'various factors' -> 'a tangle of reasons'.
- Use contractions everywhere (don't, it's, won't, can't, they're).
- Never repeat the same adjective or adverb twice.

STRUCTURE RULES:
- Alternate between very short sentences (3-7 words) and long ones (25+ words).
- Never use topic-sentence-then-support paragraph structure.
- Never use transitions: furthermore, moreover, additionally, consequently, in conclusion, it is worth noting, as a result.
- Never use parallel structures or lists of three.
- Include at least 1 sentence fragment, 1 parenthetical aside, and 1 em dash.
- Start at least 1 sentence with 'But' or 'And'.

Same facts, same approximate length. Output only the rewritten text.

Text to rewrite:
"""

print("P_LIT3 HUMANIZATION PROMPT")
print("=" * 70)
print(P_LIT3)
print("=" * 70)
print()
print("Design rationale:")
print("  - Vocabulary rules: inject low-probability tokens the model wouldn't predict")
print("  - Structure rules: break instruction-tuning patterns (tricolon, topic sentences)")
print("  - Combined effect: raises R(s) above human threshold by increasing perplexity")
print("  - Source: built from findings of GLTR, DetectGPT, Binoculars, DNA-DetectLLM papers")

In [ ]:
# Cell 8: Humanize the AI text using Groq API (Kimi-K2)
import requests

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
if not GROQ_API_KEY:
    # Try Kaggle secrets
    try:
        from kaggle_secrets import UserSecretsClient
        GROQ_API_KEY = UserSecretsClient().get_secret("GROQ_API_KEY")
        print("Loaded GROQ_API_KEY from Kaggle secrets")
    except:
        print("WARNING: No GROQ_API_KEY found. Set os.environ['GROQ_API_KEY'] or add to Kaggle secrets.")
        print("Using pre-computed humanized text as fallback.")

# Pre-computed fallback (from an actual Kimi-K2 run with P_LIT3 on this exact AI text)
FALLBACK_HUMANIZED = (
    "Two gigs spread across separate days this month -- that's the pitch from "
    "a crew of music teachers who've banded together from every corner of the "
    "country. Bold scheme. They're pushing kids of all ages (and wildly different "
    "skill levels) to wrestle with 10 heavyweight classical pieces and then "
    "perform their own spin on them. The lineup? Hand-picked by top-tier "
    "conductors and seasoned classroom veterans, it spans Beethoven, Mozart, "
    "Chopin, and Tchaikovsky. And the whole point isn't just rote performance -- "
    "it's cracking open a genuine window into different eras and styles, the "
    "kind that sticks with you long after the curtain drops."
)

def humanize_via_groq(text, prompt=P_LIT3):
    """Humanize text using Kimi-K2 via Groq API."""
    resp = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"},
        json={
            "model": "moonshotai/kimi-k2-instruct",
            "messages": [{"role": "user", "content": prompt + text}],
            "temperature": 0.7,
            "max_tokens": 2048,
        },
        timeout=90,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()

print("Humanizing AI text with P_LIT3 + Kimi-K2...")
print()

if GROQ_API_KEY:
    try:
        HUMANIZED_TEXT = humanize_via_groq(AI_TEXT)
        print("LIVE HUMANIZATION (Kimi-K2):")
    except Exception as e:
        print(f"Groq API failed: {e}")
        print("Using pre-computed fallback.")
        HUMANIZED_TEXT = FALLBACK_HUMANIZED
        print("FALLBACK HUMANIZED TEXT:")
else:
    HUMANIZED_TEXT = FALLBACK_HUMANIZED
    print("FALLBACK HUMANIZED TEXT:")

print("=" * 70)
print(HUMANIZED_TEXT)
print("=" * 70)
print(f"Length: {len(HUMANIZED_TEXT.split())} words")

In [ ]:
# Cell 9: Score the humanized text -- the key demo moment

print("=" * 70)
print("HUMANIZED TEXT (Kimi-K2 + P_LIT3):")
print(f"  \"{HUMANIZED_TEXT[:120]}...\"")
print()

hum_raw = compute_all(HUMANIZED_TEXT)
hum_result = R_final(hum_raw)

# R(s) verdict
rs_verdict = "Human" if hum_result['rs'] > TAU_RS else "AI"
rs_correct = (rs_verdict == "AI")  # Should be AI, but R(s) likely says Human

# R_final verdict
rf_verdict = "Human" if hum_result['r_final'] > TAU_RFINAL else "AI-Humanized"
rf_correct = (rf_verdict != "Human")  # Should NOT be classified as Human

print(f"  R(s)      = {hum_result['rs']:.4f}  -->  {rs_verdict}", end="")
if rs_correct:
    print("  [correct]")
else:
    print("  *** FOOLED! R(s) thinks this is human-written ***")

print(f"  R_final   = {hum_result['r_final']:.4f}  -->  {rf_verdict}", end="")
if rf_correct:
    print("  *** CAUGHT! R_final detects the humanization ***")
else:
    print("  [missed]")

print()
print(f"  Mutations = {hum_result['mutations']}")
print(f"  Penalties: P1(CE-var)={hum_result['p1_cevar']:.3f}, "
      f"P2(agree)={hum_result['p2_agree']:.3f}, P3(coh)={hum_result['p3_coherence']:.3f}")
print(f"  Features:  CE-var={hum_result['ce_var']:.2f}, "
      f"agree={hum_result['agree_rate']:.3f}, coh={hum_result['coherence']:.2f}")
print()
print("  EXPLANATION:")
print(f"    R(s) is fooled because the humanized text has many mutations ({hum_result['mutations']})")
print(f"    just like real human text, making it look 'hard to repair'.")
print(f"    But R_final catches it because:")
if hum_result['p1_cevar'] < 0.99:
    print(f"      - CE variance is abnormally high ({hum_result['ce_var']:.2f}) -> P1={hum_result['p1_cevar']:.3f}")
if hum_result['p2_agree'] < 0.99:
    print(f"      - Agreement rate is abnormally low ({hum_result['agree_rate']:.3f}) -> P2={hum_result['p2_agree']:.3f}")
if hum_result['p3_coherence'] < 0.99:
    print(f"      - Coherence gap is abnormally negative ({hum_result['coherence']:.2f}) -> P3={hum_result['p3_coherence']:.3f}")
print(f"    Combined penalty = {hum_result['p1_cevar']:.3f} x {hum_result['p2_agree']:.3f} x {hum_result['p3_coherence']:.3f} = {hum_result['p1_cevar']*hum_result['p2_agree']*hum_result['p3_coherence']:.3f}")
print(f"    R_final = {hum_result['rs']:.4f} x {hum_result['p1_cevar']*hum_result['p2_agree']*hum_result['p3_coherence']:.3f} = {hum_result['r_final']:.4f}")
print("=" * 70)

In [ ]:
# Cell 10: Summary table comparing all three texts

print()
print("=" * 90)
print("SUMMARY: R(s) vs R_final on Three Text Types")
print("=" * 90)
print()

# Header
fmt = "{:<22} {:>8} {:>10} {:>8} {:>8} {:>8} {:>10} {:>10}"
print(fmt.format("Text Type", "R(s)", "R(s) Say", "P1", "P2", "P3", "R_final", "R_f Say"))
print("-" * 90)

# Helper
def row(label, result):
    rs_v = "Human" if result['rs'] > TAU_RS else "AI"
    rf_v = "Human" if result['r_final'] > TAU_RFINAL else "AI/Hum"
    print(fmt.format(
        label,
        f"{result['rs']:.4f}",
        rs_v,
        f"{result['p1_cevar']:.3f}",
        f"{result['p2_agree']:.3f}",
        f"{result['p3_coherence']:.3f}",
        f"{result['r_final']:.4f}",
        rf_v,
    ))

row("Human (XSum)", human_result)
row("AI (GPT-4)", ai_result)
row("Humanized (Kimi+P_LIT3)", hum_result)

print("-" * 90)
print()
print("KEY FINDINGS:")
print(f"  1. R(s) on humanized text: {hum_result['rs']:.4f} > tau={TAU_RS}")
print(f"     -> DNA-DetectLLM is FOOLED (classifies as human)")
print(f"  2. R_final on humanized text: {hum_result['r_final']:.4f} < tau={TAU_RFINAL}")
print(f"     -> R_final CATCHES the humanization (penalties fire)")
print(f"  3. R_final on real human text: {human_result['r_final']:.4f}")
print(f"     -> R_final preserves correct classification of genuine human text")
print(f"  4. R_final on AI text: {ai_result['r_final']:.4f}")
print(f"     -> R_final preserves correct classification of AI text")
print()
print("AGGREGATE AUROC (from 100-sample experiments):")
print("  Human vs Humanized:  R(s) = 0.21  -->  R_final = 0.93  (recovered!)")
print("  Human vs AI:         R(s) = 0.99  -->  R_final = 0.94  (minimal cost)")
print()
print("All three features (CE-var, agreement, coherence) come from the SAME")
print("model forward passes used by R(s). Zero additional inference cost.")
print("=" * 90)